In [1]:
!ls

sample_data


In [1]:
train_json_path = '/home/carlo/MURIA_wsl/Reconocimient_Objetos/ObjectRecognition/dataset/processed/experiments/set1_balanced_subsampled/fold_1_train_coco.json'
val_json_path = '/home/carlo/MURIA_wsl/Reconocimient_Objetos/ObjectRecognition/dataset/processed/experiments/set1_balanced_subsampled/fold_1_val_coco.json'
test_json_path = '/home/carlo/MURIA_wsl/Reconocimient_Objetos/ObjectRecognition/dataset/processed/experiments/set1_balanced_subsampled/test_coco.json'

In [2]:
from dataset.efficientdet_dataset import CocoDataset
from torch.utils.data import DataLoader
from utils.data_augmentations import get_train_transforms, get_valid_transforms

/home/carlo/MURIA_wsl/Reconocimient_Objetos/ObjectRecognition/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
train_ds = CocoDataset(train_json_path, transform=get_train_transforms())
val_ds = CocoDataset(val_json_path, transform=get_valid_transforms())
test_ds = CocoDataset(test_json_path, transform=get_valid_transforms())


/home/carlo/MURIA_wsl/Reconocimient_Objetos/ObjectRecognition/.venv/lib/python3.12/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


loading annotations into memory...
Done (t=0.42s)
creating index...
index created!
loading annotations into memory...
Done (t=0.28s)
creating index...
index created!
loading annotations into memory...
Done (t=0.07s)
creating index...
index created!


In [4]:
from dataset.efficientdet_dataset import collate_fn
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=4, collate_fn=collate_fn, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=4, collate_fn=collate_fn, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=4, collate_fn=collate_fn, pin_memory=True)

/home/carlo/MURIA_wsl/Reconocimient_Objetos/ObjectRecognition/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [5]:
from models.efficientdet import EfficientDetModel
import torch

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Instanciar el modelo (3 clases: horse, penguin, pig)
my_detector = EfficientDetModel(model_name='efficientdet_d0', num_classes=3, image_size=(512, 512), bench_task='')
model_train = my_detector.get_train_model(device=device)

# 2. Optimizador (AdamW suele funcionar muy bien con EfficientDet)
optimizer = torch.optim.AdamW(model_train.parameters(), lr=2e-4, weight_decay=1e-3)

# 3. Scheduler (opcional, para bajar el LR si el entrenamiento se estanca)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

In [7]:
from tqdm import tqdm

def train_one_epoch(model, loader, optimizer, device, epoch):
    model.train()
    total_loss = 0
    
    # Envolvemos el loader con tqdm para la barra de progreso
    # 'desc' pone un texto a la izquierda, 'postfix' a la derecha
    pbar = tqdm(loader, total=len(loader), desc=f"Época {epoch+1}")
    
    for images, targets in pbar:
        # 1. Mover datos al dispositivo
        images = images.to(device).float()
        
        # 2. Preparar el padding de los targets (como vimos antes)
        max_boxes = max([t['bbox'].shape[0] for t in targets])
        batch_size = len(targets)
        
        batch_boxes = torch.zeros((batch_size, max_boxes, 4), device=device)
        batch_cls = torch.zeros((batch_size, max_boxes), device=device) - 1
        
        for i, t in enumerate(targets):
            n = t['bbox'].shape[0]
            if n > 0:
                batch_boxes[i, :n] = t['bbox']
                batch_cls[i, :n] = t['cls']

        target_res = {
            'bbox': batch_boxes,
            'cls': batch_cls
        }

        # 3. Optimización
        optimizer.zero_grad()
        loss_dict = model(images, target_res)
        loss = loss_dict['loss']
        
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        
        # 4. Actualizar la información de la barra en cada paso
        # Esto te permite ver el loss actual mientras entrena
        pbar.set_postfix(loss=f"{loss.item():.4f}", avg_loss=f"{total_loss/(pbar.n+1):.4f}")

    return total_loss / len(loader)

In [ ]:
num_epochs = 20

for epoch in range(num_epochs):
    avg_loss = train_one_epoch(model_train, train_loader, optimizer, device, epoch)
    scheduler.step()
    
    print(f"Época [{epoch+1}/{num_epochs}] - Loss Promedio: {avg_loss:.4f}")
    
    # Guardar el modelo cada 5 épocas
    if (epoch + 1) % 5 == 0:
        torch.save(my_detector.model.state_dict(), f'effdet_checkpoint_ep{epoch+1}.pth')

Época 1:   0%|          | 0/232 [00:00<?, ?it/s]/home/carlo/MURIA_wsl/Reconocimient_Objetos/ObjectRecognition/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/carlo/MURIA_wsl/Reconocimient_Objetos/ObjectRecognition/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Época 1:   0%|          | 1/232 [05:13<20:08:27, 313.88s/it, avg_loss=1.0499, loss=1.0499]
